In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from celegans_abm import default_params, simulate, density_field, clustering_metrics

o2_levels = [20, 22, 24]
times = [10, 50, 100, 150, 200]          # snapshot times (s), multiples of dt

# base = default_params()
# base.update(L=5000.0, density=60.0, T=600.0, seed=1)   # bigger box for better statistics

base = default_params()
base.update(L=5000.0, T=900.0, density=100.0, seed=1, sigma=75.0,
            Vmin=10.0, rise_c=20.0, rise_w=2.0,
            frame_dt=10.0, body_len=100.0)


results = {}
for o2 in o2_levels:
    q = dict(base, o2=o2)
    print(f"--- O2 = {o2}% ---")
    results[o2] = simulate(q, snapshots=times, verbose=False)

# ---- snapshot grid: rows = time, columns = O2 level ----
L = base["L"]
fig, axes = plt.subplots(len(times), 3, figsize=(10, 3.3 * len(times)))
for c, o2 in enumerate(o2_levels):
    for r, t in enumerate(times):
        pos, O = results[o2]["snaps"][t]
        ax = axes[r, c]
        ax.scatter(pos[:, 0], pos[:, 1], s=2, c="k")
        ax.set_xlim(0, L); ax.set_ylim(0, L); ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(f"ambient O$_2$ = {o2:.0f}%")
        if c == 0: ax.set_ylabel(f"t = {t} s")
plt.tight_layout(); plt.show()

# # ---- quantitative time evolution ----
# fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
# for o2 in o2_levels:
#     res = results[o2]
#     disp = [clustering_metrics(res["snaps"][t][0], L)["dispersion"] for t in times]
#     ax[0].plot(times, disp, "o-", label=f"{o2:.0f}%")
#     h = res["hist"]                      # columns: t, <O2>, minO2, <v>
#     ax[1].plot(h[:, 0], h[:, 3], label=f"{o2:.0f}%")
#     ax[2].plot(h[:, 0], h[:, 1], label=f"{o2:.0f}%")
# ax[0].axhline(1, c="gray", ls="--"); ax[0].set_ylabel("dispersion index (1 = random)")
# ax[1].set_ylabel("mean speed (µm/s)"); ax[2].set_ylabel("mean O$_2$ (%)")
# for a in ax: a.set_xlabel("time (s)"); a.legend()
# plt.tight_layout(); plt.show()

In [ ]:
%matplotlib inline

import matplotlib as mpl
mpl.rcParams["animation.embed_limit"] = 200

from IPython.display import HTML

from celegans_abm_repulsion_periodic_bodyviz_o2gated_new import default_params, simulate, animate_runs

o2_levels = [7.0, 15.0, 21.0]

base = default_params()

base.update(
    L=5000.0,
    T=250,      
    density=90.0,
    seed=1,
    sigma=75.0,
    frame_dt=2,
    body_len=150
)

base.update(
    r_c = 0,
    r_s = 150,
    chi=1.5,          
    rho_s=30.0,
    rho_n=2
)       
base.update(n_seg=20, body_stiff=0.6)
# runs = [simulate(dict(base, o2=o2), verbose=False) for o2 in o2_levels]

runs = [simulate(dict(base, o2=o2, o2_schedule=[(0.0, 1.0), (50.0, o2)]), verbose=False)
        for o2 in o2_levels]

anim = animate_runs(runs, [f"{o2:.0f}%" for o2 in o2_levels])


anim.save("c_elegans_oxygen_switch_2_no_body_no_repuls_new_bodies.gif", writer="pillow", fps=10)


HTML(anim.to_jshtml())

In [ ]:
import inspect, celegans_abm
print(celegans_abm.__file__)
print(inspect.signature(celegans_abm.animate_runs))